# Lab 7.3 &mdash; Structured Output, Not Prose

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Define a SCHEMA: each field's declared type and default
- Coerce a messy record -- fill missing fields, fix wrong types
- Validate that the required fields ended up present

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; Structured output, not prose](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = "/tmp/biaa-lab-07-03"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Structured output is only useful if it's **well-shaped** &mdash; every field present, every value the
right type (deck slide 7). Downstream code (route, draft, validate) indexes `rec["order_id"]` and
`rec["intent"]`, so a record that's **missing a field** or carries a **wrong-typed value** breaks the
pipeline. The fix is a **schema**: declare each field's **type** and **default**, then **coerce** any
raw record into that shape &mdash; fill missing fields from their defaults, and convert wrong types (a
numeric `order_id` becomes the **string** the orders DB is keyed by). This is real, deterministic
plumbing &mdash; the model's messy output is what you'll coerce.

In [ ]:
raw = {"order_id": 4471, "urgency": "high"}   # order_id is an int; intent & attempts are missing
print("raw (messy, from an upstream extract):", raw)
print("we want a COMPLETE, correctly-typed record out.")

## Build it
Complete `coerce` (fill defaults for missing fields; coerce wrong-typed values) and `is_valid`
(the required fields must end up present and non-None).

In [ ]:
SCHEMA = {
    "order_id": {"type": str, "default": None},
    "intent":   {"type": str, "default": "other"},
    "urgency":  {"type": str, "default": "low"},
    "attempts": {"type": int, "default": 0},
}
REQUIRED = ("order_id", "intent")

def coerce(raw):
    out = {}
    for field, spec in SCHEMA.items():
        typ, default = spec["type"], spec["default"]
        if field not in raw:
            out[field] = ___   # TODO: a missing field takes its declared default
            continue
        val = raw[field]
        if ___:   # TODO: the value is present but NOT of the declared type
            try:
                val = ___   # TODO: coerce val to the declared type
            except (ValueError, TypeError):
                val = default
        out[field] = val
    return out

def is_valid(rec):
    # required fields must be present AND non-None after coercion
    return all(___ for f in REQUIRED)   # TODO: each REQUIRED field is present and not None

try:
    print("coerced   :", coerce({"order_id": 4471, "urgency": "high"}))
    print("defaults  :", coerce({"order_id": "5090"}))
    print("valid?    :", is_valid(coerce({"order_id": "4471", "intent": "refund"})))
    print("missing id:", is_valid(coerce({"intent": "refund"})))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- A wrong-typed `order_id` (the int `4471`) is coerced to the **string** `"4471"` &mdash; the type the orders DB is keyed by.
- Missing fields fall back to their **declared defaults**, so downstream `rec["attempts"]` never `KeyError`s.
- `is_valid` is the gate: a record still missing a required field is rejected before it can break a later stage.

## Your turn (open task &mdash; no grader)
Add a new field to `SCHEMA` (e.g. `"channel": {"type": str, "default": "email"}`) and feed `coerce`
records that omit it or give it the wrong type. **What good looks like:** the coerced record always has
your field, correctly typed, and `is_valid` still guards the required set.

---
### Key takeaway
A schema of typed fields with defaults turns a messy, half-filled record into something the pipeline can trust. Coerce first, validate second -- next we produce records by extraction.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>